# IMG OPT 04 PSF Image Quality: One Case

This notebook finds Strehls from a readout corresponding to one permutation of filters/PP mask from `config/config_file_IMG_04_strehl_runs.yaml`.

Set `run_idx` below to choose which configured case to execute.

Reqs.:
- Ref. Overleaf doc IMG_OPT_04_Test_Description_PSF_Image_Quality

1. METIS-1408: Quality and alignment of the optical components within Mid-infrared ELT Imager and
Spectrograph (METIS) shall provide diffraction limited performance (Strehl ≥ 80 %)
at λ > 3μm in all modes over the entire FOV.
2. METIS-1409: The Instrument Wavefront Error (WFE) shall satisfy the diffraction limit requirement
(Strehl>0.8) at lambda = 3 μm for IMG (both LM and NQ) and IMG. The minimum
RMS WFE below shall be satisfied over the full Field Of View (FOV) relevant to the
given optical path.
3. METIS-2864: The minimum Strehl ratio of the WCU+CFO+IMG-LM optical path shall be >80% at
3.3μm over the entire field of view.
4. METIS-3503: METIS shall be able to characterise the shape of the instrument PSF across the entire
FoV using the WCU.

In [ ]:
import datetime
import logging
import os
from pprint import pprint

from modules.helpers import load_config_and_pipe
from modules.backbone import strehl_psfs

ModuleNotFoundError: No module named 'skimage'

In [ ]:
stem = "/podman-share/metis_work/playing_with_scopesim/"
observing_config_file = stem + "config/config_file_IMG_04_observing.yaml"
data_states_config_file = stem + "config/config_file_IMG_04_strehl_runs.yaml"


def _resolve_under_stem(path_value):
    if path_value.startswith("/"):
        return path_value
    return stem + path_value


def setup_logging():
    now = datetime.datetime.now()
    log_dir = stem + "IMG_04_logs/"
    log_file_name = (
        log_dir
        + "log_IMG_04_analysis_psf_image_quality_"
        + now.strftime("%Y-%m-%d_%H-%M-%S")
        + ".txt"
    )
    os.makedirs(log_dir, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file_name),
            logging.StreamHandler(),
        ],
        force=True,
    )
    return log_file_name


def build_data_states(data_states_config):
    defaults = data_states_config.get("defaults", {})
    runs = data_states_config.get("runs", [])

    data_states = []
    for entry in runs:
        merged = {**defaults, **entry}
        merged["file_name"] = _resolve_under_stem(merged["file_name"])
        merged["config_coords_guesses_file_name"] = _resolve_under_stem(
            merged["config_coords_guesses_file_name"]
        )
        merged["results_write_dir"] = _resolve_under_stem(
            merged["results_write_dir"]
        )
        data_states.append(merged)

    return data_states

In [ ]:
log_file_name = setup_logging()
print(f"Logging to: {log_file_name}")

observing_config = load_config_and_pipe(
    config_file_choice=observing_config_file,
    print_one_line=False,
)
data_states_config = load_config_and_pipe(
    config_file_choice=data_states_config_file,
    print_one_line=False,
)

data_states = build_data_states(data_states_config)

for idx, state in enumerate(data_states):
    print(
        f"[{idx}] filter={state['filter_name']}, "
        f"fp_mask={state['fp_mask']}, "
        f"pp_mask={state['pp_mask']}, "
        f"results={state['results_write_dir']}"
    )

In [ ]:
run_idx = 0
state = data_states[run_idx]

print(f"Selected run index: {run_idx}")
pprint(state)

In [ ]:
strehl_psfs(
    state["file_name"],
    fp_mask=state["fp_mask"],
    pp_mask=state["pp_mask"],
    filter_name=state["filter_name"],
    fit_simmed_psf=state["fit_simmed_psf"],
    fit_annular_aperture_free=state["fit_annular_aperture_free"],
    fit_annular_aperture_fixed=state["fit_annular_aperture_fixed"],
    psfs_subset=state["psfs_subset"],
    config_coords_guesses_file_name=state["config_coords_guesses_file_name"],
    config_observing=observing_config,
    results_write_dir=state["results_write_dir"],
    fit_method=state.get("fit_method", "curve_fit"),
)

print("Finished single-case Strehl analysis.")
print(f"Results written under: {state['results_write_dir']}")